In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark=SparkSession.builder.appName("PySpark Ingestion Standardization").getOrCreate()

In [2]:
customer_file="customers.csv"
order_file="orders.csv"

In [3]:
def read_file(file_path):
  df=spark.read.csv(
      file_path,
      header=True,
      inferSchema=True
  )
  return df

In [5]:
customers_df=read_file(customer_file)
orders_df=read_file(order_file)

In [6]:
customers_df.show()
orders_df.show()

+-----------+-------------+---------------+
|Customer ID|Customer Name|          Email|
+-----------+-------------+---------------+
|        101|        Rahul|rahul@gmail.com|
|        102|        Priya|priya@gmail.com|
|        103|        Arjun|arjun@gmail.com|
|        104|        Sneha|sneha@gmail.com|
|        105|        Kiran|kiran@gmail.com|
+-----------+-------------+---------------+

+--------+-----------+------------+
|Order ID|Customer ID|Order Amount|
+--------+-----------+------------+
|    1001|        101|      1200.5|
|    1002|        102|       850.0|
|    1003|        101|      450.75|
|    1004|        103|      2100.0|
|    1005|        105|      675.25|
|    1006|        104|       990.0|
+--------+-----------+------------+



In [7]:
customers_df.printSchema()

root
 |-- Customer ID: integer (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Email: string (nullable = true)



In [8]:
orders_df.printSchema()

root
 |-- Order ID: integer (nullable = true)
 |-- Customer ID: integer (nullable = true)
 |-- Order Amount: double (nullable = true)



In [9]:
def standardize_column_name(df):
  for column in df.columns:
    new_column=column.strip().lower().replace(" ","_")
    df=df.withColumnRenamed(column , new_column)
  return df

In [10]:
customers_df=standardize_column_name(customers_df)
orders_df=standardize_column_name(orders_df)

In [11]:
print(customers_df.columns)

['customer_id', 'customer_name', 'email']


In [12]:
print(orders_df.columns)

['order_id', 'customer_id', 'order_amount']


In [13]:
customers_df=customers_df.select(
    col("customer_id").cast("integer").alias("customer_id"),
    col("customer_name").cast("string").alias("customer_name"),
    col("email").cast("string").alias("email")
)

In [15]:
orders_df=orders_df.select(
    col("order_id").cast("integer").alias("order_id"),
    col("customer_id").cast("integer").alias("customer_id"),
    col("order_amount").cast("double").alias("order_amount")
)

In [16]:
customers_df.show()
orders_df.show()

+-----------+-------------+---------------+
|customer_id|customer_name|          email|
+-----------+-------------+---------------+
|        101|        Rahul|rahul@gmail.com|
|        102|        Priya|priya@gmail.com|
|        103|        Arjun|arjun@gmail.com|
|        104|        Sneha|sneha@gmail.com|
|        105|        Kiran|kiran@gmail.com|
+-----------+-------------+---------------+

+--------+-----------+------------+
|order_id|customer_id|order_amount|
+--------+-----------+------------+
|    1001|        101|      1200.5|
|    1002|        102|       850.0|
|    1003|        101|      450.75|
|    1004|        103|      2100.0|
|    1005|        105|      675.25|
|    1006|        104|       990.0|
+--------+-----------+------------+



In [17]:
customers_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)



In [18]:
orders_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_amount: double (nullable = true)



In [19]:
customers_df.filter(
    col("customer_id").isNull()
).show()

+-----------+-------------+-----+
|customer_id|customer_name|email|
+-----------+-------------+-----+
+-----------+-------------+-----+



In [20]:
orders_df.filter(
    col("customer_id").isNull()
).show()

+--------+-----------+------------+
|order_id|customer_id|order_amount|
+--------+-----------+------------+
+--------+-----------+------------+



In [21]:
customers_df.count()

5

In [22]:
orders_df.count()

6